In [ ]:
# データ読み込み
import category_encoders as ce
import numpy as np
import pandas as pd

In [ ]:
# programs.csvを読み込み
programs_df = pd.read_csv("data/programs.csv")

In [ ]:
print(f"programs.csvの形状: {programs_df.shape}")
print("\nカラム一覧:")
print(programs_df.columns.tolist())
print("\n先頭5行:")
programs_df.head()

In [ ]:
# データの詳細情報を確認
print("=== データ型情報 ===")
print(programs_df.dtypes)

print("\n=== 欠損値情報 ===")
print(programs_df.isnull().sum())

print("\n=== 基本統計量 ===")
print(programs_df.describe())

print("\n=== データ期間 ===")
# 年・月・日を結合して日付型に変換
programs_df["開催日"] = pd.to_datetime(
    programs_df[["年", "月", "日"]].astype(str).agg("-".join, axis=1)
)

print(f"開始日: {programs_df['開催日'].min().date()}")
print(f"終了日: {programs_df['開催日'].max().date()}")

print("\n=== レース場数 ===")
print(f"レース場数: {programs_df['レース場番号'].nunique()}")
print(f"レース場一覧: {sorted(programs_df['レース場番号'].unique())}")

print("\n=== ユニークなレース数 ===")
unique_races = (
    programs_df.groupby(["年", "月", "日", "レース場番号", "レース番号"])
    .size()
    .shape[0]
)
print(f"総レース数: {unique_races}")
print(f"総艇数: {len(programs_df)}")
print(f"1レースあたりの平均艇数: {len(programs_df) / unique_races:.1f}")

In [ ]:
def make_race_id(row):
    """
    年月日とレース場番号、レース番号から一意なIDを作成する関数
    """
    date_str = f"{row['年']:04d}{row['月']:02d}{row['日']:02d}"
    place_str = f"{row['レース場番号']:02d}"
    race_str = f"{row['レース番号']:02d}"
    return int(f"{date_str}{place_str}{race_str}")

In [ ]:
programs_df.insert(0, "レースID", programs_df.apply(make_race_id, axis=1))

In [ ]:
programs_df.head(12)

In [ ]:
# 級別を数値に変換
def grade_to_numeric(grade):
    if grade == "A1":
        return 3
    elif grade == "A2":
        return 2
    elif grade == "B1":
        return 1
    elif grade == "B2":
        return 0
    else:
        return np.nan


programs_df["級別"] = programs_df["級別"].apply(grade_to_numeric)

In [ ]:
programs_df.head(12)

In [ ]:
feature_columns = [
    "レースID",
    "枠番",
    "選手登番",
    "年齢",
    "体重",
    "級別",
    "全国勝率",
    "全国2連率",
    "当地勝率",
    "当地2連率",
    "モーター2連率",
    "ボート2連率",
]
programs_df = programs_df[feature_columns]

In [ ]:
programs_df.head(12)

In [ ]:
# レース内平均差を追加
def calc_rate_diff(rates):
    rates_numeric = pd.to_numeric(rates, errors="coerce")
    avg_rate = rates_numeric.mean()
    return rates_numeric - avg_rate

In [ ]:
  # レース内全国勝率差
  programs_df["レース内全国勝率差"] = programs_df.groupby("レースID")[
      "全国勝率"
  ].transform(calc_rate_diff)
programs_df.head(12)

In [ ]:
print("\n=== レースIDの先頭20行 ===")
print(programs_df.head(20))

In [ ]:
print("=== データ型情報 ===")
print(programs_df.dtypes)

In [ ]:
# results.csvを読み込み
results_df = pd.read_csv("data/results.csv")

In [ ]:
print(f"results.csvの形状: {results_df.shape}")
print("\nカラム一覧:")
print(results_df.columns.tolist())
print("\n先頭5行:")
results_df.head()

In [ ]:
results_df.insert(0, "レースID", results_df.apply(make_race_id, axis=1))

In [ ]:
print("\n=== レースIDの先頭20行 ===")
print(results_df.head(20))

In [ ]:
# programs_dfとresults_dfをマージして着順情報を追加
# レースIDと選手登番でマージ
merged_df = programs_df.merge(
    results_df[["レースID", "選手登番", "着順"]],
    on=["レースID", "選手登番"],
    how="left",
)

print(f"マージ前のprograms_df形状: {programs_df.shape}")
print(f"マージ後のmerged_df形状: {merged_df.shape}")

# 着順が取得できたかどうか確認
print(f"\n着順が取得できた件数: {merged_df['着順'].notna().sum()}")
print(f"着順が取得できなかった件数: {merged_df['着順'].isna().sum()}")

# programs_dfを更新
programs_df = merged_df.copy()

print(f"\n最終的なprograms_df形状: {programs_df.shape}")
print("\n着順が追加されたデータの先頭5行:")
programs_df.head()

In [ ]:
# 最後に選手登番を削除
programs_df.drop(columns=["選手登番"], inplace=True)

In [ ]:
# train.csvとして保存
programs_df.to_csv("data/train.csv", index=False, encoding="utf-8-sig")
print("\n=== train.csvとして保存完了 ===")
print(f"保存先: data/train.csv")